# Data Loading

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()

while not (PROJECT_ROOT / "src").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate the project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    
from src.detector.data import (
    build_grouped_volume_registry,
    load_e2e_volume,
    select_bscan,
)

E2E_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "heyex"
    / "meta"
)

registry = build_grouped_volume_registry(
    e2e_directory=E2E_DIRECTORY,
    progression_groups={
        "fast": [8, 9, 12, 41, 49],
        "slow": [17, 23, 35, 36, 47],
    },
)

record = registry[0]

volume = load_e2e_volume(
    record.e2e_path
)

bscan_index, bscan = select_bscan(
    volume,
    selection="center",
)

print("Subject:", record.subject_id)
print("Group:", record.progression_group)
print("Volume shape:", volume.shape)
print("Selected B-scan:", bscan_index)
print("B-scan shape:", bscan.shape)

# Preprocessing

In [ ]:
from src.detector.preprocessing import (
    preprocess_bscan,
)

# --------------------------------------------------
# Preprocessing configuration
# --------------------------------------------------

PREPROCESSING_CONFIG = {
    "layer_name": "BM",

    # Flattening
    "reference_row": None,
    "flatten_fill_value": 0.0,

    # Crop
    "depth_below_layer": 150,
    "include_boundary": True,
    "require_full_depth": False,
    "crop_fill_value": 0.0,

    # Normalization
    "normalization_method": "zscore",
    "lower_percentile": 1.0,
    "upper_percentile": 99.0,

    # Denoising
    "denoise_method": "gaussian",
    "gaussian_sigma": (1.0, 0.5),
}

# --------------------------------------------------
# Run preprocessing
# --------------------------------------------------

result = preprocess_bscan(
    volume=volume,
    bscan_index=bscan_index,
    **PREPROCESSING_CONFIG,
)

print("Completed preprocessing.")
print(result)

In [ ]:
fig, axes = plt.subplots(
    1,
    5,
    figsize=(20,5),
)

axes[0].imshow(
    result.raw_bscan,
    cmap="gray",
    aspect="auto",
)
axes[0].set_title("Raw")

axes[1].imshow(
    result.flattened_bscan,
    cmap="gray",
    aspect="auto",
)
axes[1].set_title("Flattened")

axes[2].imshow(
    result.sub_layer_crop,
    cmap="gray",
    aspect="auto",
)
axes[2].set_title("Crop")

axes[3].imshow(
    result.normalized_scan,
    cmap="gray",
    aspect="auto",
)
axes[3].set_title("Normalized")

axes[4].imshow(
    result.denoised_scan,
    cmap="gray",
    aspect="auto",
)
axes[4].set_title("Denoised")

plt.tight_layout()

In [ ]:
from pprint import pprint

pprint(result.metadata)